In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import pdfplumber

def import_bank_transactions(file_path, file_type='csv'):
    """
    Import transactions from various file formats
    """
    if file_type == 'csv':
        df = pd.read_csv(file_path)
    elif file_type == 'excel':
        df = pd.read_excel(file_path)
    elif file_type == 'pdf':
        df = extract_from_pdf(file_path)
    
    # Standardize column names
    column_mapping = {
        'Date': 'date',
        'Transaction Date': 'date',
        'Description': 'description',
        'Memo': 'description',
        'Amount': 'amount',
        'Debit': 'debit',
        'Credit': 'credit'
    }
    
    df.rename(columns=column_mapping, inplace=True)
    
    # Convert date to datetime
    df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.date
    
    # Combine debit/credit into single amount column if needed
    if 'debit' in df.columns and 'credit' in df.columns:
        df['amount'] = df['credit'].fillna(0) - df['debit'].fillna(0)
        df.drop(['debit', 'credit'], axis=1, inplace=True)
    
    # Clean description field
    df['description'] = df['description'].str.strip().str.upper()
    
    return df.sort_values('date').reset_index(drop=True)


def extract_from_pdf(pdf_path):
    """
    Extract transaction data from PDF bank statements
    """
    transactions = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            # Parse text based on your bank's format
            # This is a simplified example
            lines = text.split('\n')
            for line in lines:
                # Custom parsing logic for your bank's format
                parts = line.split()
                if len(parts) >= 3:
                    transactions.append({
                        'date': parts[0],
                        'description': ' '.join(parts[1:-1]),
                        'amount': parts[-1]
                    })
    
    return pd.DataFrame(transactions)

# Import transactions from multiple sources
bank_transactions = import_bank_transactions('D://test.xlsx', 'excel') # EXCEL
credit_card = extract_from_pdf("D://test.pdf") # EXCEL

# Combine all transactions
df = pd.concat([bank_transactions, credit_card], ignore_index=True)
print(f"Total transactions imported: {len(df)}")

Total transactions imported: 6


In [4]:
bank_transactions

,date,description,amount
0,2026-04-16,INTERNET,1
1,2026-04-18,SLACK,1
2,2026-04-20,UBER,100
3,2026-04-22,RENT,-100


In [5]:
credit_card

,date,description,amount
0,26-01-02,Expense office supplies,1500
1,26-01-03,Income services,1200


In [3]:
df

,date,description,amount
0,2026-04-16,INTERNET,1
1,2026-04-18,SLACK,1
2,2026-04-20,UBER,100
3,2026-04-22,RENT,-100
4,26-01-02,Expense office supplies,1500
5,26-01-03,Income services,1200


In [7]:
def create_categorization_rules():
    """
    Define rules for automatic transaction categorization
    """
    rules = {
        'Office Supplies': ['STAPLES', 'OFFICE DEPOT', 'AMAZON'],
        'Utilities': ['ELECTRIC', 'GAS COMPANY', 'WATER', 'INTERNET'],
        'Marketing': ['GOOGLE ADS', 'FACEBOOK', 'LINKEDIN', 'MAILCHIMP'],
        'Software': ['MICROSOFT', 'ADOBE', 'DROPBOX', 'SLACK'],
        'Meals & Entertainment': ['RESTAURANT', 'COFFEE', 'STARBUCKS', 'LUNCH'],
        'Travel': ['AIRLINE', 'HOTEL', 'UBER', 'LYFT', 'RENTAL CAR'],
        'Professional Services': ['LEGAL', 'ACCOUNTING', 'CONSULTING'],
        'Rent': ['LANDLORD', 'PROPERTY MANAGEMENT', 'RENT'],
        'Insurance': ['INSURANCE'],
        'Bank Fees': ['MONTHLY FEE', 'SERVICE CHARGE', 'ATM FEE'],
        'Client Payment': ['PAYMENT RECEIVED', 'INVOICE', 'DEPOSIT'],
        'Owner Draw': ['TRANSFER TO PERSONAL', 'OWNER WITHDRAWAL']
    }
    return rules

# DONE TILL HERE

In [8]:
rules = create_categorization_rules()
print(rules)

{'Office Supplies': ['STAPLES', 'OFFICE DEPOT', 'AMAZON'], 'Utilities': ['ELECTRIC', 'GAS COMPANY', 'WATER', 'INTERNET'], 'Marketing': ['GOOGLE ADS', 'FACEBOOK', 'LINKEDIN', 'MAILCHIMP'], 'Software': ['MICROSOFT', 'ADOBE', 'DROPBOX', 'SLACK'], 'Meals & Entertainment': ['RESTAURANT', 'COFFEE', 'STARBUCKS', 'LUNCH'], 'Travel': ['AIRLINE', 'HOTEL', 'UBER', 'LYFT', 'RENTAL CAR'], 'Professional Services': ['LEGAL', 'ACCOUNTING', 'CONSULTING'], 'Rent': ['LANDLORD', 'PROPERTY MANAGEMENT', 'RENT'], 'Insurance': ['INSURANCE'], 'Bank Fees': ['MONTHLY FEE', 'SERVICE CHARGE', 'ATM FEE'], 'Client Payment': ['PAYMENT RECEIVED', 'INVOICE', 'DEPOSIT'], 'Owner Draw': ['TRANSFER TO PERSONAL', 'OWNER WITHDRAWAL']}


In [9]:
df['category'] = 'uncategorized'

In [10]:
df['confidence'] = 0.0

In [11]:
df

,date,description,amount,category,confidence
0,2026-04-16,INTERNET,1,uncategorized,0.0
1,2026-04-18,SLACK,1,uncategorized,0.0
2,2026-04-20,UBER,100,uncategorized,0.0
3,2026-04-22,RENT,-100,uncategorized,0.0
4,26-01-02,Expense office supplies,1500,uncategorized,0.0
5,26-01-03,Income services,1200,uncategorized,0.0


In [52]:
for category, keywords in rules.items():
    print(keywords)
    for keyword in keywords:
        mask = df['description'].str.contains(keyword, case=False, na=False)
        #print(mask)
        # Only categorize if not already categorized
        uncategorized = df['category'] == 'uncategorized'
  
        df.loc[mask & uncategorized, 'category'] = category
        df.loc[mask & uncategorized, 'confidence'] = 0.9

['STAPLES', 'OFFICE DEPOT', 'AMAZON']
['ELECTRIC', 'GAS COMPANY', 'WATER', 'INTERNET']
['GOOGLE ADS', 'FACEBOOK', 'LINKEDIN', 'MAILCHIMP']
['MICROSOFT', 'ADOBE', 'DROPBOX', 'SLACK']
['RESTAURANT', 'COFFEE', 'STARBUCKS', 'LUNCH']
['AIRLINE', 'HOTEL', 'UBER', 'LYFT', 'RENTAL CAR']
['LEGAL', 'ACCOUNTING', 'CONSULTING']
['LANDLORD', 'PROPERTY MANAGEMENT', 'RENT']
['INSURANCE']
['MONTHLY FEE', 'SERVICE CHARGE', 'ATM FEE']
['PAYMENT RECEIVED', 'INVOICE', 'DEPOSIT']
['TRANSFER TO PERSONAL', 'OWNER WITHDRAWAL']


In [53]:
df

,date,description,amount,category,confidence
0,2026-04-16,INTERNET,1,Utilities,0.9
1,2026-04-18,SLACK,1,Software,0.9
2,2026-04-20,UBER,100,Travel,0.9
3,2026-04-22,RENT,-100,Rent,0.9
4,26-01-02,Expense office supplies,1500,uncategorized,0.0
5,26-01-03,Income services,1200,uncategorized,0.0


In [56]:
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
# Categorize by amount patterns
df.loc[(df['amount'] > 0) & (df['category'] == 'uncategorized'), 'category'] = 'Income'
df.loc[(df['amount'] < 0) & (df['category'] == 'uncategorized'), 'category'] = 'Other Expense'

In [57]:
df

,date,description,amount,category,confidence
0,2026-04-16,INTERNET,1,Utilities,0.9
1,2026-04-18,SLACK,1,Software,0.9
2,2026-04-20,UBER,100,Travel,0.9
3,2026-04-22,RENT,-100,Rent,0.9
4,26-01-02,Expense office supplies,1500,Income,0.0
5,26-01-03,Income services,1200,Income,0.0


In [ ]:
def categorize_transactions(df, rules):
    """
    Automatically categorize transactions based on description
    """
    df['category'] = 'Uncategorized'
    df['confidence'] = 0.0
    
    for category, keywords in rules.items():
        for keyword in keywords:
            mask = df['description'].str.contains(keyword, case=False, na=False)
            # Only categorize if not already categorized
            uncategorized = df['category'] == 'Uncategorized'
            df.loc[mask & uncategorized, 'category'] = category
            df.loc[mask & uncategorized, 'confidence'] = 0.9
    
    # Categorize by amount patterns
    df.loc[(df['amount'] > 0) & (df['category'] == 'Uncategorized'), 'category'] = 'Income'
    df.loc[(df['amount'] < 0) & (df['category'] == 'Uncategorized'), 'category'] = 'Other Expense'
    
    return df